# Assignments 1 & 2

In [ ]:
# Required packages
import pandas as pd
from scipy import stats
from scipy.stats.mstats import mquantiles

## Data Cleaning

In [ ]:
# Import data
transaction = pd.read_csv("../Data/transaction_data.csv")

# Dataframe format
transaction.columns = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]

# Transaction data type fix
transaction["transaction_date"] = pd.to_datetime(transaction["transaction_date"], format = '%Y-%m-%d')

# Fix lead time
transaction[transaction["lead_time"] == 2800]
transaction.loc[transaction["lead_time"] == 2800, "lead_time"] = 28

# Unit price and order quantity values fix
transaction["unit_price"] = transaction["unit_price"].abs()
transaction["order_quantity"] = transaction["order_quantity"].abs()

# NA values
Transaction_ValuesMissing = ["sku_number","inventory_type", "stocking_type", "manufacturing_site", "division_code"] 
transaction[Transaction_ValuesMissing] = transaction[Transaction_ValuesMissing].fillna("Missing")
transaction

## Getting Country from Manufacturing Site

In [ ]:
country_values = {"US":"United States", "PL":"Poland", "CA":"Canada", "JP":"Japan"}

transaction["country"] = transaction["manufacturing_site"].str.slice(0, 2).map(country_values)
transaction["country"] = transaction["country"].fillna("Missing")

transaction

## Unique SKUs

In [ ]:
transaction_FG_MTS = transaction.query('inventory_type == "FG" and stocking_type == "MTS"')
transaction_FG_MTS["sku_number"].drop_duplicates().count()
transaction_FG_MTS["sku_number"].unique()

## Unique Manufacturing Sites

In [ ]:
transaction_FG_MTS["manufacturing_site"].drop_duplicates().count()
transaction_FG_MTS["manufacturing_site"].unique()

## Unique Divisions

In [ ]:
transaction_FG_MTS["division_code"].drop_duplicates().count()
transaction_FG_MTS["division_code"].unique()

## Transactions by Order Quantity

### Top 10

In [ ]:
transaction_FG_MTS.sort_values("order_quantity", ascending = False).head(10)

### Bottom 10

In [ ]:
transaction_FG_MTS.sort_values("order_quantity").head(10)

## Transactions by Sales Value

In [ ]:
transaction_FG_MTS.insert(9, "sales_value", transaction["order_quantity"] * transaction["unit_price"])

### Top 10

In [ ]:
transaction_FG_MTS.sort_values("sales_value", ascending = False).head(10)

### Bottom 10

In [ ]:
transaction_FG_MTS.sort_values("sales_value").head(10)

## Filtering for Make To Stock SKUs

In [ ]:
mts = transaction.query('stocking_type in "MTS"')

# Create new data frame grouped by sku and aggregated quantity and lead time for SS calculation
mts_df = (mts
         .groupby("sku_number")
         .agg(min_qty = ("order_quantity", "min"),
              max_qty = ("order_quantity", "max"),
              avg_qty = ("order_quantity", "mean"),
              median_qty = ("order_quantity", "median"),
              variance_qty = ("order_quantity", "var"),
              std_qty = ("order_quantity", "std"),
              avg_lead_time = ("lead_time", "mean"))
        .reset_index())
mts_df

## Safety Stock Calculation

In [ ]:
# Set the z-score for each service level, under a normal distribution
mts_z75 = stats.norm.ppf(.75)
mts_z90 = stats.norm.ppf(.90)
mts_z95 = stats.norm.ppf(.95)

print(mts_z75, mts_z90, mts_z95)

In [ ]:
# Add column for safety stock at 75% service level
mts_df["safety_stock_75"] = (
    mts_z75
    * mts_df["std_qty"]
    * mts_df["avg_lead_time"] ** 0.5
)
mts_df

In [ ]:
# Add column for safety stock at 90% service level
mts_df["safety_stock_90"] = (
    mts_z90
    *mts_df["std_qty"]
    *mts_df["avg_lead_time"]** 0.5 
)
mts_df

In [ ]:
# Add column for safety stock at 95% service level
mts_df["safety_stock_95"] = (
    mts_z95
    *mts_df["std_qty"]
    *mts_df["avg_lead_time"]** 0.5 
)
mts_df

## Safety Stock Distribution

### Largest

In [ ]:
mts_df.sort_values(by = "safety_stock_95", ascending = False).head(1)

### Smallest

In [ ]:
mts_df.sort_values(by = "safety_stock_95").head(1)

### Average

In [ ]:
# Mean of safety stock column across all SKUs
safety_75 = mts_df["safety_stock_75"].mean()
safety_90 = mts_df["safety_stock_90"].mean()
safety_95 = mts_df["safety_stock_95"].mean()
print(safety_75, ",", safety_90, ",", safety_95)

## Demand by Quantile

In [ ]:
daily_order_qty = mts.groupby(["sku_number", "transaction_date"], as_index=False)["order_quantity"].sum()
daily_order_qty

In [ ]:
quantiles = pd.DataFrame(columns = ["sku_number", "safety_stock_75", "safety_stock_90", "safety_stock_95"])
orders = daily_order_qty[["sku_number", "order_quantity"]]
for sku in orders["sku_number"].unique():
	prob = mquantiles(orders.query('sku_number == @sku')["order_quantity"], prob = [0.75, 0.9, 0.95], alphap = 0, betap = 1)
	quantiles.loc[len(quantiles)] = [sku, prob[0], prob[1], prob[2]]
quantiles.sort_values("safety_stock_95", ascending = False)

# References:
- Bobbitt, Z. (2021, April 26). Percentile vs. quartile vs. quantile: What’s the difference? Statology. https://www.statology.org/percentile-vs-quartile-vs-quantile/

We used this to understand quantiles and how we can relate it to safety stock calculation. 

- Stack Overflow(2022). "How do I calculate square root in Python?". Retrieved November 17, 2026. https://stackoverflow.com/questions/70793490/how-do-i-calculate-square-root-in-python 

When searching for "How to square root in python" on Stack Overflow, we come across a post (Stack Overflow, 2022) that has last been edited this year on August 24th that indicates the snippet below:

```
Option 2: Fractional exponent
The power operator (**) or the built-in pow() function can also be used to calculate a square root. Mathematically speaking, the square root of a equals a to the power of 1/2.

The power operator requires numeric types and matches the conversion rules for binary arithmetic operators, so in this case it will return either a float or a complex number

>>> 9 ** (1/2)
3.0
>>> 9 ** .5  # Same thing
3.0
>>> 2 ** .5
1.4142135623730951

```